# NHL Data Pipeline — Players

In [ ]:
import requests
import sqlite3
import pandas as pd
import time

DB_PATH = "../data/nhl.db"
ROSTER_URL = "https://api-web.nhle.com/v1/roster/{team}/current"
TIMEOUT = 15
SLEEP = 0.4

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

teams = pd.read_sql("SELECT team_id, team_abbrev FROM teams ORDER BY team_abbrev", conn)

In [ ]:
def flatten_roster(roster, team_id):
    players = []
    for group in ("forwards", "defensemen", "goalies"):
        for p in roster.get(group, []):
            players.append({
                "player_id": p["id"],
                "team_id": team_id,
                "first_name": p["firstName"]["default"],
                "last_name": p["lastName"]["default"],
                "position": p.get("positionCode"),
                "jersey_number": p.get("sweaterNumber"),
                "birth_date": p.get("birthDate"),
                "birth_country": p.get("birthCountry"),
                "height_cm": p.get("heightInCentimeters"),
                "weight_kg": p.get("weightInKilograms"),
                "shoots_catches": p.get("shootsCatches"),
                "headshot_url": p.get("headshot"),
            })
    return players

In [ ]:
all_players, failed_teams = [], []

for row in teams.itertuples(index=False):
    try:
        resp = requests.get(ROSTER_URL.format(team=row.team_abbrev), timeout=TIMEOUT)
        resp.raise_for_status()
        all_players.extend(flatten_roster(resp.json(), row.team_id))
    except Exception as e:
        failed_teams.append((row.team_abbrev, str(e)))
    time.sleep(SLEEP)

players = pd.DataFrame(all_players).drop_duplicates(subset="player_id")
print(f"{len(players)} players collected, {len(failed_teams)} team requests failed")

In [ ]:
for row in players.itertuples(index=False):
    cur.execute(
        "INSERT OR IGNORE INTO players (player_id, team_id, first_name, last_name, position, "
        "jersey_number, birth_date, birth_country, height_cm, weight_kg, shoots_catches, headshot_url) "
        "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        row,
    )
conn.commit()
conn.close()